# vector-normalize-keepdim — worked example 2: Normalize query vectors for cosine attention

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `vector-normalize-keepdim`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In cosine-similarity attention, both query and key vectors are L2-normalized before the dot product. This is done with `.norm(dim=-1, keepdim=True)`, which collapses the head-dimension while preserving the tensor rank so the subsequent division broadcasts correctly over that dimension.

## Worked solution

**Step 1 — understand the shape.**
Queries `q` have shape `(batch, heads, seq, d_head)`. We want to normalize along `d_head` — the last axis. After normalization, each `(d_head,)` vector should have unit length.

**Step 2 — compute norms with keepdim.**
We call `q.norm(dim=-1, keepdim=True)`, which produces shape `(batch, heads, seq, 1)`. The `keepdim=True` ensures we don't lose the fourth dimension, which is needed for the division to broadcast over `d_head`.

**Step 3 — divide.**
Dividing `q` (shape `(B, H, S, D)`) by the norms (shape `(B, H, S, 1)`) broadcasts the scalar norm across the D dimension for each vector independently. Each `d_head`-length slice now lies on the unit hypersphere.

In [ ]:
import torch as t

t.manual_seed(7)
B, H, S, D = 2, 4, 6, 16  # batch, heads, seq_len, d_head

q = t.randn(B, H, S, D)

# Normalize along d_head so each query vector is unit length
q_norms = q.norm(dim=-1, keepdim=True)  # (B, H, S, 1)
q_unit = q / q_norms                    # (B, H, S, D)

print('q_unit shape:', q_unit.shape)
result_norms = q_unit.norm(dim=-1)      # (B, H, S)
print('Sample norms (first head, first batch):', result_norms[0, 0].tolist())
print('All close to 1.0:', t.allclose(result_norms, t.ones_like(result_norms), atol=1e-6))